# 📉 Gradient Descent for Linear and Logistic Regression

<a href="https://colab.research.google.com/github/rubenfonnegra/machine_learning/blob/master/Sem_09/GD_linear_logistic_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> 
<a href="https://github.com/rubenfonnegra/machine_learning/blob/master/Sem_09/GD_linear_logistic_regression.ipynb" target="_parent"><img src="https://img.shields.io/badge/%E2%80%8B-Open%20in%20Github-blue?logo=github" alt="Open In Github"/></a> 


In this notebook, we will implement gradient descent manually using only NumPy for the optimization process.

### Learning objectives

By the end of this notebook, you will be able to:

- Explain gradient descent as an iterative optimization algorithm.
- Identify parameters, predictions, loss, gradients, and learning rate.
- Derive and implement gradients for linear regression.
- Derive and implement gradients for logistic regression.
- Observe how parameters and loss change during optimization.
- Build regression and classification models optimized by GD

### Documentation

- [```sympy```](https://www.sympy.org/es/) 

---

> **📘 Machine Learning**  
> **Author:** Rubén D. Fonnegra, Ph.D. \
> **Institution:** Institución Universitaria Pascual Bravo  
> © 2026 · Educational use with attribution

## Imports

In [ ]:
#@markdown #### **🛠️⚙️📦 Install complementary dependencies**. 

from tqdm.auto import tqdm
import subprocess, time, sys

LIB = "MLTools-1.2-py3-none-any.whl"
URL = "https://drive.google.com/uc?id=18Y834Tvtj_-Px9yNmbAB4yV4L0gcIZ20"

commands = [
    ("📦 Downloading resources", ["gdown", URL, "-O", LIB], 35),
    ("🔧 Installing dependencies", [sys.executable, "-m", "pip", "install", "-q", LIB], 55),
    ("🧹 Finishing", ["rm", "-f", LIB], 10)
]

print("⚙️ Iniciando configuración del entorno...\n")

try:
    with tqdm(total=100, desc="Preparando", bar_format="{desc}: {bar} {n:.0f}%") as bar:
        for label, command, weight in commands:
            bar.set_description(label)
            subprocess.run(command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
            for _ in range(weight):
                time.sleep(0.01)
                bar.update(1)

    print("\n✅ Configuración completada correctamente.")

except subprocess.CalledProcessError as e:
    print("\n❌ Error durante la configuración")
    print(f"Exit code: {e.returncode}")

    if e.stdout:
        print("\n📤 STDOUT:")
        print(e.stdout)

    if e.stderr:
        print("\n🔍 STDERR:")
        print(e.stderr)

    print("\n❌ No fue posible configurar el entorno. Ejecute nuevamente la celda.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sympy as sym
from sklearn.model_selection import train_test_split
from IPython.display import clear_output

from MLTools import generate_linear_regression, generate_isotropic_blobs

## What is Gradient Descent?

Suppose a model contains parameters collected in $\theta$, and a loss function $\mathcal{L}(\theta)$ tells us how wrong the model is.

Gradient descent repeatedly updates the parameters in the direction that **reduces the loss**:

$$
\theta_{next}=\theta_{current}-\alpha\nabla \mathcal{L}(\theta)
$$

where:

- $\theta$ = model parameters,
- $\mathcal{L}$ = loss function,
- $\nabla \mathcal{L}(\theta)$ = gradient of the loss,
- $\alpha$ = learning rate.

The gradient tells us the direction of the steepest **increase** in the loss. Therefore, we move in the opposite direction.

### The basic algorithm

1. Initialize the parameters.
2. Make predictions.
3. Compute the loss.
4. Compute the gradients.
5. Update the parameters.
6. Repeat until the loss stops improving or the maximum number of epochs is reached.

## Part 1: Linear Regression

For one feature, our model is:

$$
\hat{y}_i=wx_i+b
$$

We want to find $w$ and $b$ that minimize Mean Squared Error:

$$
\mathcal{L}(w,b)=\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2
$$

## Generate a linear regression dataset

In [ ]:
X, y, y_true = generate_linear_regression(
    n_samples = _ ,
    x_range = (-5, 5),
    slope = 3.5,
    intercept = 2.0,
    noise = 2.0,
    random_state = 42
)

plt.figure(figsize=(9, 5))
plt.scatter(X[:, 0], y, alpha=0.7, label="Observed data")
plt.plot(X[:, 0], y_true, linewidth=2, label="Underlying function")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Linear Regression Data")
plt.legend()
plt.show()

## Holdout


In [ ]:
X_train, X_test, y_train, y_test = train_test_split( _ , _ , test_size = 0.20, random_state = 1)

print("Training:", X_train.shape, y_train.shape)
print("Testing:", X_test.shape, y_test.shape)

## Define prediction and loss functions

In [ ]:
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

## Deriving the gradients

The loss is:

$$
\mathcal{L}(w,b)=\frac{1}{n}\sum_{i=1}^{n}(y_i-(wx_i+b))^2
$$

The partial derivative with respect to $w$ is:

$$
\frac{\partial \mathcal{L}}{\partial w}
=
\frac{2}{n}\sum_{i=1}^{n}(\hat{y}_i-y_i) \times x_i
$$

The partial derivative with respect to $b$ is:

$$
\frac{\partial \mathcal{L}}{\partial b}
=
\frac{2}{n}\sum_{i=1}^{n}(\hat{y}_i-y_i)
$$

The updates are therefore:

$$
w_{next} \leftarrow w_{current}-\alpha\frac{\partial \mathcal{L}}{\partial w}
$$

$$
b_{next} \leftarrow b_{current}-\alpha\frac{\partial \mathcal{L}}{\partial b}
$$

In [ ]:
Xs, ys, ws, bs = sym.symbols('Xs, ys, ws, bs')

der_loss = ((ws*Xs + bs) - ys)**2
print ("Derivative L w.r.t. w --> ", sym.Derivative(der_loss, ws, evaluate=True))
print ("Derivative L w.r.t. b --> ", sym.Derivative(der_loss, bs, evaluate=True))

## One gradient step

In [ ]:
def linear_gradients(X, y, w, b):
    predictions = X[:, 0] _ 
    errors = predictions - y
    dw = (2 / len(X)) * np.sum(X[:, 0] * errors)
    db = (2 / len(X)) * np.sum(errors)
    return dw, db

w, b = 0.0, 0.0
dw, db = linear_gradients(X_train, y_train, w, b)

print("Initial w:", w)
print("Initial b:", b)
print("Gradient for w:", round(dw, 4))
print("Gradient for b:", round(db, 4))

## Optimize Linear Regression with Gradient Descent

In [ ]:
learning_rate = 0.01
epochs = 500
w, b = 0.0, 0.0

linear_loss_history = []
linear_parameter_history = []

for epoch in range(epochs):
    predictions = X_train[:, 0] _ 
    loss = _ 
    dw, db = linear_gradients(X_train, y_train, w, b)

    w -= _ 
    b -= _ 

    linear_loss_history.append(loss)
    linear_parameter_history.append((w, b))

    if epoch % 50 == 0:
        print(f"Epoch {epoch:3d} | MSE: {loss:.4f} | w: {w:.4f} | b: {b:.4f}")
        plt.scatter(X_train, y_train, alpha=0.8,)
        # line between min-max-points
        h_ = [ min(X_train), max(X_train) ]
        v_ = [ w * min(X_train) + b, w * max(X_train) + b ]
        plt.title('Epoch: {0}, MSE: {1}'.format(epoch, loss))
        plt.plot(h_, v_, "r", linewidth=3, alpha=0.6, label="Gradient Descent model")
        plt.show(), plt.pause(1), clear_output(wait=False)

    if max(abs(learning_rate * dw), 
           abs(learning_rate * db) ) < 1e-3:
        break

print(f"Final parameters -> w={w:.4f}, b={b:.4f}")

## Visualize the optimized regression model

In [ ]:
x_line = np.linspace(X[:, 0].min(), X[:, 0].max(), 200)
y_line = w * x_line + b

plt.figure(figsize=(9, 5))
plt.scatter(X_train[:, 0], y_train, alpha=0.55, label="Training data")
plt.scatter(X_test[:, 0], y_test, alpha=0.8, label="Test data")
plt.plot(x_line, y_line, linewidth=3, label="Gradient Descent model")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Linear Regression after Gradient Descent")
plt.legend()
plt.show()

## Observe convergence

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(linear_loss_history)
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Linear Regression - Loss During Optimization")
plt.show()

test_predictions = X_test[:, 0] _ 
print("Training MSE:", round(mse(y_train, X_train[:, 0]), 4))
print("Test MSE:", round(mse(y_test, test_predictions), 4))

# 11. What does the learning rate do?

The learning rate $\alpha$ controls the size of each optimization step.

- **Too small:** optimization may be very slow.
- **Appropriate:** loss decreases steadily.
- **Too large:** gradient descent may overshoot the minimum or diverge.

Let's compare several learning rates using exactly the same data and initial parameters.

In [ ]:
learning_rates = [0.001, 0.01, 0.05]

plt.figure(figsize=(9, 5))

for lr in learning_rates:
    current_w, current_b = 0.0, 0.0
    history = []

    for epoch in range(200):
        predictions = X_train[:, 0] _ 
        history.append(mse(y_train, predictions))
        dw, db = linear_gradients(X_train, y_train, current_w, current_b)
        current_w -= lr * dw
        current_b -= lr * db

    plt.plot(history, linestyle='--', label=f"learning rate={lr}")

plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Effect of the Learning Rate")
plt.legend()
plt.show()

## Part II — Logistic Regression

Linear regression predicts unrestricted numerical values. For binary classification, we want a probability between 0 and 1.

We first compute a linear score:

$$
z=\mathbf{X}\mathbf{w}+b
$$

and transform it with the **sigmoid function**:

$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$

Therefore:

$$
P(y=1|\mathbf{x})=\sigma(\mathbf{X}\mathbf{w}+b)
$$

A probability of at least $0.5$ will be classified as class 1.

## Generate a binary classification dataset

In [ ]:
X_class, y_class = generate_isotropic_blobs(
    n_samples = _ ,
    centers = [(-2.0, -1.5), (2.0, 1.5)],
    cluster_std = 1.6,
    random_state = 42
)

plt.figure(figsize=(8, 6))
plt.scatter(X_class[:, 0], X_class[:, 1], c=y_class, edgecolors="black", alpha=0.7, cmap='Paired')
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Binary Classification Dataset")
plt.show()

## Holdout

In [ ]:
Xc_train, Xc_test, yc_train, yc_test = train_test_split( _ , _ , test_size = 0.20, random_state = 1)

print("Training:", Xc_train.shape, yc_train.shape)
print("Testing:", Xc_test.shape, yc_test.shape)

## Standardize the classification features manually

Gradient descent usually behaves better when features have comparable scales.

For each feature:

$$
z=\frac{x-\mu}{\sigma}
$$

The mean and standard deviation are calculated **only from the training data**.

In [ ]:
feature_mean = Xc_train.mean(axis=0)
feature_std = Xc_train.std(axis=0)

Xc_train_scaled = (Xc_train - feature_mean) / feature_std
Xc_test_scaled = (Xc_test - feature_mean) / feature_std

print("Training means:", np.round(Xc_train_scaled.mean(axis=0), 4))
print("Training standard deviations:", np.round(Xc_train_scaled.std(axis=0), 4))

## Sigmoid and probability prediction

In [ ]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def logistic_predict_proba(X, weights, bias):
    return sigmoid(X @ weights + bias)

def logistic_predict(X, weights, bias, threshold=0.5):
    probabilities = logistic_predict_proba(X, weights, bias)
    return (probabilities >= threshold).astype(int)

## Binary Cross-Entropy

For logistic regression, Mean Squared Error is not the usual optimization objective. We use **Binary Cross-Entropy**:

$$
J(\mathbf{w},b)=
-\frac{1}{n}
\sum_{i=1}^{n}
\left[
y_i\log(\hat{p}_i)
+
(1-y_i)\log(1-\hat{p}_i)
\right]
$$

A good model assigns high probability to the correct class, producing a smaller loss.

In [ ]:
def binary_cross_entropy(y_true, probabilities):
    eps = 1e-12
    probabilities = np.clip(probabilities, eps, 1 - eps)
    return -np.mean(
        y_true * np.log(probabilities)
        + (1 - y_true) * np.log(1 - probabilities)
    )

## Logistic Regression gradients

Let:

$$
\hat{\mathbf{p}}=\sigma(\mathbf{X}\mathbf{w}+b)
$$

For Binary Cross-Entropy, the gradients simplify to:

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{w}}
=
\frac{1}{n}\mathbf{X}^{T}(\hat{\mathbf{p}}-\mathbf{y})
$$

$$
\frac{\partial \mathcal{L}}{\partial b}
=
\frac{1}{n}\sum_{i=1}^{n}(\hat{p}_i-y_i)
$$

The same gradient-descent rule applies:

$$
\mathbf{w_{next}}\leftarrow\mathbf{w_{current}}-\alpha\frac{\partial \mathcal{L}}{\partial\mathbf{w}}
$$

$$
b_{next}\leftarrow b_{current}-\alpha\frac{\partial \mathcal{L}}{\partial b}
$$

In [ ]:
Xs, ys, ws, bs = sym.symbols('Xs, ys, ws, bs')

z = ws*Xs + bs
y_pred = 1 / (1 + sym.exp(-z))
der_loss = -(ys*sym.log(y_pred) + (1-ys)*sym.log(1-y_pred))

print("Derivative L w.r.t. w -->", sym.Derivative(der_loss, ws, evaluate=True).simplify())
print("Derivative L w.r.t. b -->", sym.Derivative(der_loss, bs, evaluate=True).simplify())

## One gradient step


In [ ]:
def logistic_gradients(X, y, weights, bias):
    probabilities = logistic_predict_proba(X, weights, bias)
    errors = probabilities - y
    dw = (X.T @ errors) / len(X)
    db = np.mean(errors)
    return dw, db

weights = np.zeros(Xc_train_scaled.shape[1])
bias = 0.0
dw, db = logistic_gradients(Xc_train_scaled, yc_train, weights, bias)

print("Initial weights:", weights)
print("Initial bias:", bias)
print("Weight gradients:", np.round(dw, 4))
print("Bias gradient:", round(db, 4))

## Optimize Logistic Regression with Gradient Descent

In [ ]:
learning_rate = 0.1
epochs = 1000
weights = np.zeros(Xc_train_scaled.shape[1])
bias = 0.0
logistic_loss_history = []

for epoch in range(epochs):
    probabilities = logistic_predict_proba(Xc_train_scaled, weights, bias)
    loss = _ 
    dw, db = logistic_gradients(Xc_train_scaled, yc_train, weights, bias)

    weights -= _ 
    bias -= _ 
    logistic_loss_history.append(loss)

    if epoch % 50 == 0:
        predictions = logistic_predict(Xc_train_scaled, weights, bias)
        accuracy = np.mean(predictions == yc_train)
        print(f"Epoch {epoch:4d} | BCE: {loss:.4f} | Accuracy: {accuracy:.4f}")

        plt.figure(figsize=(8, 6))
        plt.scatter(Xc_train_scaled[:, 0], Xc_train_scaled[:, 1], c=yc_train, edgecolors="black", alpha=0.7, cmap='Paired' )
        # Decision boundary:
        x1 = np.linspace( Xc_train_scaled[:, 0].min(), Xc_train_scaled[:, 0].max(), 100 )
        x2 = -(weights[0] * x1 + bias) / weights[1]

        plt.plot( x1, x2, c='r', linewidth=3, alpha=0.6, label="Decision boundary" )
        plt.show(), plt.pause(1), clear_output(wait=False)

    
    if max(abs(np.max(learning_rate * dw)), 
           abs(np.max(learning_rate * db)) ) < 1e-3:
        break

print("Final weights:", np.round(weights, 4))
print("Final bias:", round(bias, 4))

## Evaluate the classifier

In [ ]:
train_predictions = logistic_predict(Xc_train_scaled, weights, bias)
test_predictions = logistic_predict(Xc_test_scaled, weights, bias)

train_accuracy = np.mean(train_predictions == yc_train)
test_accuracy = np.mean(test_predictions == yc_test)

print("Training accuracy:", round(train_accuracy, 4))
print("Test accuracy:", round(test_accuracy, 4))
print("Final BCE:", round(logistic_loss_history[-1], 4))

## Visualize the Logistic Regression decision boundary

In [ ]:
x1_min, x1_max = Xc_train_scaled[:, 0].min() - 0.5, Xc_train_scaled[:, 0].max() + 0.5
x2_min, x2_max = Xc_train_scaled[:, 1].min() - 0.5, Xc_train_scaled[:, 1].max() + 0.5

xx, yy = np.meshgrid(
    np.linspace(x1_min, x1_max, 300),
    np.linspace(x2_min, x2_max, 300)
)
grid = np.column_stack((xx.ravel(), yy.ravel()))
grid_predictions = logistic_predict(grid, weights, bias).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, grid_predictions, alpha=0.25, cmap='Paired')
plt.scatter( Xc_train_scaled[:, 0], Xc_train_scaled[:, 1], c=yc_train, edgecolors="black", alpha=0.7, cmap='Paired' )
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Logistic Regression — Decision Boundary")
plt.show()

## Observe Logistic Regression convergence

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(logistic_loss_history)
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy")
plt.title("Logistic Regression — Loss During Optimization")
plt.show()

## Same optimizer, different models

The two models look different, but gradient descent follows the same logic.

| Step | Linear Regression | Logistic Regression |
|---|---|---|
| Linear score | $\mathbf{Xw}+b$ | $\mathbf{Xw}+b$ |
| Output | Numerical value | Sigmoid probability |
| Loss | MSE | Binary Cross-Entropy |
| Parameters | $\mathbf{w},b$ | $\mathbf{w},b$ |
| Gradient | Derivative of MSE | Derivative of BCE |
| Update | $\theta-\alpha\nabla \mathcal{L}(\theta)$ | $\theta-\alpha\nabla \mathcal{L}(\theta)$ |

Gradient descent is therefore **not a regression algorithm or classification algorithm**. It is an **optimization strategy** that can be used to minimize many differentiable objective functions.

## Final challenge

Modify both implementations and investigate:

1. What happens when the learning rate is extremely small?
2. What happens when it is too large?
3. Change the initial parameter values. Do the models reach similar solutions?
4. Add a stopping criterion based on the change in loss.
5. For logistic regression, change the classification threshold from `0.5` to `0.3` and `0.7`.
6. Generate a more difficult dataset and observe how the loss and final model change.

In [ ]:
# Your code here

---

## 📄 Attribution

This notebook was developed by **Rubén D. Fonnegra** as educational material for Machine Learning courses at **Institución Universitaria Pascual Bravo**.
You may use, share, and adapt this material for educational purposes, provided that appropriate credit is given to the original author.

**Suggested citation:**
> Fonnegra Tarazona, R. D. (2026). *Gradient descent - Linear and logistic regression: Machine Learning Notebook*. Institución Universitaria Pascual Bravo.